# 예제 03. 결과 분석 도구 모음
빅데이터프로그래밍 · 13주차

## 목표
- 학습 곡선을 해석한다
- 틀린 사례를 분류해 원인을 찾는다
- 모델의 한계와 개선 방향을 정리한다

프로젝트 어느 주제에나 쓸 수 있는 분석 코드를 모아 두었습니다. 필요한 셀만 골라 복사하세요.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 학습 곡선 진단
곡선 모양을 자동으로 판정합니다.


In [ ]:
def diagnose(history):
    """history: [(train_loss, train_acc, val_loss, val_acc), ...]"""
    tr_l = [h[0] for h in history]; tr_a = [h[1] for h in history]
    va_l = [h[2] for h in history]; va_a = [h[3] for h in history]

    gap = tr_a[-1] - va_a[-1]
    best = int(np.argmin(va_l)) + 1
    rising = va_l[-1] > min(va_l) * 1.05

    print(f"최종 학습 정확도 {tr_a[-1]:.4f} / 검증 정확도 {va_a[-1]:.4f}")
    print(f"차이 {gap:.4f}")
    print(f"검증 손실 최소 epoch {best} / 전체 {len(history)}")
    print()

    if gap > 0.10 and rising:
        print("진단: 과적합 — 학습 데이터를 외우고 있습니다")
        print("  → Dropout · 데이터 증강 · 조기 종료 (9주차)")
    elif tr_a[-1] < 0.7:
        print("진단: 과소적합 — 아직 덜 배웠습니다")
        print("  → 모델을 키우기 · epoch 늘리기 · 학습률 확인")
    elif np.std(va_a[-5:]) > 0.03:
        print("진단: 불안정 — 검증 정확도가 흔들립니다")
        print("  → 학습률 낮추기 · batch 크기 늘리기")
    else:
        print("진단: 정상적으로 학습됐습니다")
        if best < len(history) * 0.7:
            print(f"  → 다만 epoch {best} 에서 멈추는 편이 나았습니다")


# 예시 데이터로 확인
fake = [(0.9-i*0.08, 0.5+i*0.05, 0.9-i*0.05+max(0,(i-5))*0.06, 0.5+i*0.035-max(0,(i-5))*0.01)
        for i in range(10)]
diagnose(fake)


## 2. 예측 결과 모으기 — 재사용 함수


In [ ]:
def collect(model, loader, device=device):
    """모델의 모든 예측을 모아 돌려줍니다"""
    model.eval()
    xs, ys, ps, probs = [], [], [], []
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(device))
            prob = torch.softmax(out, dim=1).cpu()
            xs.append(x); ys.append(y)
            ps.append(out.argmax(dim=1).cpu()); probs.append(prob)
    return (torch.cat(xs), torch.cat(ys), torch.cat(ps), torch.cat(probs))


print("collect(model, loader) → (입력, 정답, 예측, 확률)")


## 3. 틀린 사례를 원인별로 나누기


In [ ]:
def error_breakdown(true, pred, prob, class_names=None):
    wrong = (pred != true).nonzero().flatten()
    conf = prob.max(dim=1).values

    rows = [
        {"구분": "전체", "개수": len(true), "비율": 1.0},
        {"구분": "맞음", "개수": int((pred == true).sum()),
         "비율": round(float((pred == true).float().mean()), 4)},
        {"구분": "틀림", "개수": len(wrong),
         "비율": round(len(wrong)/len(true), 4)},
        {"구분": "  확신하고 틀림 (>0.9)", "개수": int((conf[wrong] > 0.9).sum()),
         "비율": round(float((conf[wrong] > 0.9).float().mean()), 4) if len(wrong) else 0},
        {"구분": "  애매해서 틀림 (<0.6)", "개수": int((conf[wrong] < 0.6).sum()),
         "비율": round(float((conf[wrong] < 0.6).float().mean()), 4) if len(wrong) else 0},
    ]
    return pd.DataFrame(rows)


# 예시
true = torch.randint(0, 10, (500,))
pred = true.clone(); pred[:80] = torch.randint(0, 10, (80,))
prob = torch.rand(500, 10); prob = prob / prob.sum(dim=1, keepdim=True)
error_breakdown(true, pred, prob)


**확신하고 틀린 것**이 많으면 모델이 잘못 배운 것입니다. **애매해서 틀린 것**은 데이터 자체가 어려운 경우가 많습니다.


## 4. 클래스별 정확도와 혼동 쌍


In [ ]:
def per_class_table(true, pred, class_names):
    rows = []
    for c, name in enumerate(class_names):
        m = true == c
        if m.sum() == 0: continue
        rows.append({"클래스": name, "개수": int(m.sum()),
                     "정확도": round(float((pred[m] == c).float().mean()), 4)})
    df = pd.DataFrame(rows).sort_values("정확도")
    return df


def top_confusions(true, pred, class_names, k=5):
    n = len(class_names)
    cm = torch.zeros(n, n, dtype=torch.int32)
    for t, p in zip(true, pred):
        cm[t, p] += 1
    off = cm.clone(); off.fill_diagonal_(0)
    idx = off.flatten().argsort(descending=True)[:k]
    rows = []
    for f in idx:
        t, p = divmod(f.item(), n)
        rows.append({"정답": class_names[t], "예측": class_names[p], "횟수": int(off[t, p])})
    return pd.DataFrame(rows)


CLASSES = [f"클래스{i}" for i in range(10)]
print(per_class_table(true, pred, CLASSES).head().to_string(index=False))
print()
print(top_confusions(true, pred, CLASSES).to_string(index=False))


## 5. 회귀 문제용 — 시계열 프로젝트라면


In [ ]:
def regression_report(pred, true):
    err = pred - true
    mae = np.abs(err).mean()
    rmse = np.sqrt((err**2).mean())
    rng = true.max() - true.min()

    print(f"MAE  {mae:.5f}")
    print(f"RMSE {rmse:.5f}")
    print(f"값 범위 {true.min():.3f} ~ {true.max():.3f}")
    print(f"범위 대비 RMSE {rmse/rng*100:.2f}%")

    fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
    ax[0].plot(true, label="실제", linewidth=1.4)
    ax[0].plot(pred, label="예측", linewidth=1.2, linestyle="--")
    ax[0].set_title("실제 vs 예측"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(err, linewidth=1); ax[1].axhline(0, color="crimson", linestyle="--")
    ax[1].set_title("오차"); ax[1].grid(alpha=.3)
    ax[2].scatter(true, pred, s=6, alpha=.5)
    lims = [min(true.min(), pred.min()), max(true.max(), pred.max())]
    ax[2].plot(lims, lims, color="crimson", linestyle="--")
    ax[2].set_xlabel("실제"); ax[2].set_ylabel("예측"); ax[2].set_title("산점도")
    plt.tight_layout(); plt.show()


t = np.sin(np.arange(200) * 0.1)
p = t + np.random.randn(200) * 0.08
regression_report(p, t)


## 6. 한계와 개선 방향 정리 양식
발표 자료에 그대로 옮길 수 있는 형태로 만듭니다.


In [ ]:
limits = pd.DataFrame([
    {"한계": "특정 클래스에서 정확도가 낮다",
     "근거": "클래스별 정확도 표 — 셔츠 0.72",
     "개선 방향": "해당 클래스 데이터를 늘리거나 가중치를 조정"},
    {"한계": "학습-검증 차이가 0.08 있다",
     "근거": "학습 0.95 / 검증 0.87",
     "개선 방향": "Dropout 확률을 높이거나 증강을 강화"},
    {"한계": "확신하고 틀린 사례가 30건",
     "근거": "확률 0.9 이상인데 오답",
     "개선 방향": "해당 이미지를 확인 — 레이블 오류 가능성"},
])
print(limits.to_string(index=False))


한계를 적을 때는 **반드시 숫자 근거**를 함께 적으세요. "성능이 조금 아쉽다"는 분석이 아닙니다.


## 직접 해보기
1. 자기 프로젝트의 history로 `diagnose` 를 돌려 보세요.
2. `error_breakdown` 결과를 보고 확신하고 틀린 사례를 직접 확인하세요.
3. 한계 표를 자기 프로젝트에 맞게 채우세요.


In [ ]:
# 여기에 작성하세요
